# Visualizations of the EEG data in the TF domain

This notebook is used for different basic visualisations in the TF domain

Author: Magnus Evensen, Malte Færgemann Lau $\\$
Project: Bachelor's Project - EEG Social Interaction

In [ ]:
import h5py
import numpy as np
import mne
import os
import pandas as pd
import glob
import gc
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib as mpl

# mpl setup
mpl.rcParams['image.cmap'] = 'ocean'
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.Set2(np.linspace(0, 1, 8)))
# plt.rcParams['axes.prop_cycle'] = plt.cycler(color = [(225/255,102/255,102/255), (82/255,158/255,205/255),  (194/255,226/255,170/255), (225/255,178/255,102/255), (178/255,102/255,225/255),(225/255,225/255,102/255),(102/255,102/255,225/255)])
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['figure.figsize'] = (6, 4)
mpl.rcParams['lines.linewidth'] = 1
mpl.rcParams['figure.dpi'] = 400
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
dtu_colors = {'dtured' : (0.6,0,0), 'navyblue' : (0.0118,0.0588,0.3098), 'red' : (0.9098,0.2471,0.2824), 'blue': (0.1843,0.2431,0.9176), 'green' : (0,0.5333,0.2078), 'orange' : (0.9882,0.4627,0.2039), 'purple' : (0.4745,0.1373,0.5569), 'brightgreen' : (0.1216,0.8157,0.5098), 'yellow' : (0.9647,0.8157,0.3019), 'pink' : (0.9686,0.7333,0.6941), 'grey' : (0.8549,0.8549,0.8549), 'red' : (0.9098,0.2471,0.2824)}
mne.set_log_level('WARNING')

In [ ]:
# path = os.path.abspath('../FG_Data/FG_overview_df_v2.pkl') # Locals
path = "/work3/s224188/FG_Data/FG_overview_df_v2.pkl"      # HPC
overview = pd.read_pickle(path)
overview

In [ ]:
# Create df for plots
# folder_path = os.path.abspath('../FG_Data/PreprocessedEEGData') # Local
folder_path = "/work3/s224188/FG_Data/PreprocessedEEGData/"     # HPC
# Use glob to find all FIF files in the folder
epoch_files = glob.glob(os.path.join(folder_path, '*-epo.fif'))
epochs_list = []
# Load each FIF file into a list of epoch objects
for f in epoch_files:
    # Extract participant ID from filename
    basename = os.path.basename(f)
    # For filenames like '301A_FG_preprocessed-epo.fif',
    # this splits into ['301A', 'FG', 'preprocessed-epo.fif']
    Exp_id = basename.split('_')[0]
    
    # Lookup gender in your DataFrame based on the participant_id
    gender = overview.loc[overview['Exp_id'] == Exp_id, 'Gender'].iloc[0]
    friend = overview.loc[overview['Exp_id'] == Exp_id, 'Friend_status'].iloc[0]
    
    # Read the epochs
    epochs = mne.read_epochs(f, preload=False, verbose=False)
    
    # Store everything in a dictionary (or tuple) in the list
    epochs_list.append({
        'Exp_id': Exp_id,
        'Gender': gender,
        'Epochs': epochs,
        'Friend': friend
    })
epochs_df = pd.DataFrame(epochs_list)
sfreq = int(epochs_df['Epochs'][0].info['sfreq'])
n_epochs, n_channels, n_timepoints = epochs_df['Epochs'][0].get_data().shape
epochs_df

In [ ]:
epochs_df[epochs_df['Exp_id'] == '326A']

In [ ]:
epochs_df['Epochs'][0].get_data().shape

In [ ]:
def roof(x):
    return int(x) + 1 if x > int(x) else int(x)

In [ ]:
def get_psds_for_channel(participants_data, channel, settings, window, band = [1,40], overlap = 0.75, mean = True):
    
    overlap = int(window * sfreq * overlap) # scale overlap to window size
    psd_dict = {}
    for setting in settings:
        all_psds = []
        for epochs in tqdm(participants_data, desc=f'For condition: {setting}, for channel: {channel} with freq band: {band}'):
            # Pick channel 
            filtered_epochs = epochs.copy().load_data().pick([channel])
            filtered_epochs = filtered_epochs.filter(l_freq=band[0], h_freq=band[1], fir_design='firwin', n_jobs=10, verbose = False)
            filtered_epochs = filtered_epochs[setting]
            # Compute PSD
            psds = filtered_epochs.compute_psd(method='welch', window = 'hamming', average = None, 
                                               fmin=band[0], fmax=band[1], n_fft=int(sfreq*window), 
                                               n_overlap=overlap, n_jobs=10, verbose = False);
            if mean:
                mean_psds = np.array(psds.get_data().mean(axis=0).mean(axis = 1)).flatten()
                # Standardize to baseline 
                baseline_mean = mean_psds[int(len(mean_psds)/24):int(len(mean_psds)/12)].mean() # Dividing len(mean_psds) with 24 and 12 gives the index of -0.25s and 0s
                baseline_corrected = mean_psds / baseline_mean
                all_psds.append(baseline_corrected) # append the mean over epochs and frequencies to all psds
            else:
                psds_list = np.array(psds.get_data().mean(axis=0))[0]
                baseline_snippet = psds_list[:, int(len(psds_list[0])/24):int(len(psds_list[0])/12) + 1]
                baseline_mean = np.mean(baseline_snippet, axis=1, keepdims=True)
                baseline_corrected_dB = 10*np.log10(psds_list / baseline_mean)
                all_psds.append(baseline_corrected_dB)
            # Unload data from iteration to save memory space
            del filtered_epochs
            gc.collect()
            
        label = f'{setting}, {band} frequencies'
        psd_dict[label] = np.reshape(all_psds, (len(all_psds), len(all_psds[0]))) if mean else np.array(all_psds).mean(axis=0)
        
    return psd_dict

In [ ]:
def get_psds_for_channel_all_settings(participants_data, channel, window, band = [1,40], overlap = 0.75, mean = True):
    
    overlap = int(window * sfreq * overlap) # scale overlap to window size
    psd_dict = {}
    all_psds = []
    for epochs in tqdm(participants_data, desc=f'For all conditions, for channel: {channel} with freq band: {band}'):
        # Pick channel 
        filtered_epochs = epochs.copy().load_data().pick([channel])
        filtered_epochs = filtered_epochs.filter(l_freq=band[0], h_freq=band[1], fir_design='firwin', n_jobs=10, verbose = False)
        # Compute PSD
        psds = filtered_epochs.compute_psd(method='welch', window = 'hamming', average = None, 
                                               fmin=band[0], fmax=band[1], n_fft=int(sfreq*window), 
                                               n_overlap=overlap, n_jobs=10, verbose = False);
        if mean:
            mean_psds = np.array(psds.get_data().mean(axis=0).mean(axis = 1)).flatten()
            # Standardize to baseline 
            baseline_mean = mean_psds[int(len(mean_psds)/24):int(len(mean_psds)/12)].mean() # Dividing len(mean_psds) with 24 and 12 gives the index of -0.25s and 0s
            baseline_corrected = mean_psds / baseline_mean
            all_psds.append(baseline_corrected) # append the mean over epochs and frequencies to all psds
        else:
            psds_list = np.array(psds.get_data().mean(axis=0))[0]
            baseline_snippet = psds_list[:, int(len(psds_list[0])/24):int(len(psds_list[0])/12) + 1]
            baseline_mean = np.mean(baseline_snippet, axis=1, keepdims=True)
            baseline_corrected_dB = 10*np.log10(psds_list / baseline_mean)
            all_psds.append(baseline_corrected_dB)
        # Unload data from iteration to save memory space
        del filtered_epochs
        gc.collect()
            
        label = f'All conditions, {band} frequencies'
        psd_dict[label] = np.reshape(all_psds, (len(all_psds), len(all_psds[0]))) if mean else np.array(all_psds).mean(axis=0)
        
    return psd_dict

In [ ]:
window = 1
channel = 'C3'
settings = ['T1P']
overlap = 0.9
spectrogram_ex= get_psds_for_channel(participants_data = [epochs_df['Epochs'][0]], window=window, channel=channel, settings=settings, overlap=overlap, mean=False)

In [ ]:
list(spectrogram_ex.values())[0]

In [ ]:
window = 1
channel = 'C3'
settings = ['T1P', 'T3P', 'T1Pn', 'T3Pn']
overlap = 0.9
spectrogram = get_psds_for_channel(participants_data = epochs_df['Epochs'], window=window, channel=channel, settings=settings, overlap=overlap, mean=False)

In [ ]:
list(spectrogram.values())[0].shape

In [ ]:
mpl.rcParams['image.cmap'] = 'bwr'
def plot_spectrogram(psd_list, freqs = np.arange(1,41)):
    extreme = max(abs(np.min(psd_list)), abs(np.max(psd_list)))
    fig = plt.figure(figsize=(10,4))
    n_times = psd_list.shape[1]
    time_vector = np.linspace(-0.5, 5.5, n_times)
    '''start_idx = np.argmin(np.abs(time_vector - 0))
    end_idx = np.argmin(np.abs(time_vector - 4))
    psd_list = psd_list[:, start_idx:end_idx+1]
    time_vector = time_vector[start_idx:end_idx+1]'''
    
    plt.imshow(psd_list, aspect='auto', origin='lower',
                extent=[time_vector[0], time_vector[-1], freqs[0], freqs[-1]], vmin=-extreme, vmax=extreme)
    plt.colorbar(label='Power relative to baseline (dB)')
    plt.title('Spectrogram')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_spectrogram(list(spectrogram_ex.values())[0])

In [ ]:
plot_spectrogram(list(spectrogram.values())[1] - list(spectrogram.values())[3])

In [ ]:
def plot_TF(dict, mean = False, window = 1, overlay = True): 
    """ Takes a dict where each value should be of size (n_segments, n_channels), \\
    and returns frequency over time plots in a (n_items, 2) grid"""
    times = np.linspace(-0.5, 5.5, len(next(iter(dict.values()))[0])) #+  window*sfreq/2/len(next(iter(dict.values()))[0]) # Create time points equal to powers
    
    if overlay: 
        fig, ax = plt.subplots(1,1, figsize= (12,6))
    else: 
        fig, axes = plt.subplots(roof(len(dict)/2), 2, figsize=(12, len(dict)*2), constrained_layout = True)
        axes = axes.flatten()
    
    
    for i, (data_label, powers) in enumerate(dict.items()):
        if not overlay:
            ax = axes[i]
        if mean: # Plot mean with std
            mean_line = np.mean(powers, axis = 0)
            peak_ERD = min(mean_line)
            std_line = np.std(powers, axis = 0)/np.sqrt(len(powers))
            ax.plot(times, mean_line, label = data_label + f'.  Peak ERD: {round((1-peak_ERD)*100,1)}%')
            ax.fill_between(times, mean_line - std_line, mean_line + std_line, alpha=0.3)
            ax.axhline(y=peak_ERD, color=['green', 'orange', 'blue', 'pink'][i], linestyle='--', alpha = 0.7) # plot baseline
        
        else: # Plot each participant
            for power in powers:
                ax.plot(times, power, alpha = 0.5)
        if not overlay: 
            ax.axhline(y=1, color='black', linestyle='--', label='baseline', alpha = 0.3) # plot baseline
            ax.set_title(f"{data_label}")
            ax.set_ylim(0.55,1.35)
            ax.set_xlim(-0.1,4)
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Power')
    if overlay:
        ax.axhline(y=1, color='black', linestyle='--', label='baseline', alpha = 0.3) # plot baseline
        ax.set_title("ERD/S visualization")
        ax.set_ylim(0.50,1.35)
        ax.set_xlim(-0.1,4)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Power')
        ax.legend()
    plt.show()

In [ ]:
window = 1
channel = 'C3'
settings = ['T1P', 'T1Pn']
overlap = 0.9
psds = get_psds_for_channel(participants_data=epochs_df['Epochs'], overlap=overlap, channel=channel, settings=settings, band = [8,12], window=3/8)

In [ ]:
plot_TF(psds, mean=True)

In [ ]:
overlap = 0.9
settings = ['T1P', 'T1Pn', 'T3P', 'T3Pn']
channel = 'C3'
Alpha_psds = get_psds_for_channel(participants_data=epochs_df['Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
Beta_psds = get_psds_for_channel(participants_data=epochs_df['Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
#Theta_psds = get_psds_for_channel(participants_data=epochs_df['Epochs'], channel=channel, settings=settings, band = [4,8], window=3/4, overlap = overlap)
#unfiltered_psds = get_psds_for_channel(participants_data=epochs_df['Epochs'], channel=channel, settings=settings, window=1, overlap = overlap)

In [ ]:
plot_TF(Alpha_psds, mean = True)

In [ ]:
plot_TF(Beta_psds, mean = True)

In [ ]:
plot_TF(Theta_psds, mean = True)

In [ ]:
plot_TF(unfiltered_psds, mean = True)

In [ ]:
overlap = 0.9
settings = ['T1P']
channel = 'C3'
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, band = [8,12], window=3/8, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, band = [8,12], window=3/8, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_alpha = F_psds | nF_psds
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, band = [12,35], window=3/12, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, band = [12,35], window=3/12, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_beta = F_psds | nF_psds
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, band = [4,8], window=3/4, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, band = [4,8], window=3/4, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_theta = F_psds | nF_psds
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, band = [1,40], window=3/3, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel_all_settings(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, band = [1,40], window=3/3, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_all = F_psds | nF_psds


In [ ]:
plot_TF(F_alpha, mean=True)

In [ ]:
plot_TF(F_beta, mean=True)

In [ ]:
plot_TF(F_theta, mean=True)

In [ ]:
plot_TF(F_all, mean=True)

In [ ]:
overlap = 0.9
channel = 'C3'
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, settings=['T3P'], band = [8,12], window=3/8, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, settings=['T3P'], band = [8,12], window=3/8, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_gf_alpha = F_psds | nF_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, settings=['T3P'], band = [12,35], window=3/12, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, settings=['T3P'], band = [12,35], window=3/12, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_gf_beta = F_psds | nF_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, settings=['T1P'], band = [8,12], window=3/8, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, settings=['T1P'], band = [8,12], window=3/8, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_sf_alpha = F_psds | nF_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, settings=['T1P'], band = [12,35], window=3/12, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, settings=['T1P'], band = [12,35], window=3/12, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_sf_beta = F_psds | nF_psds

In [ ]:
plot_TF(F_gf_alpha, mean=True)

In [ ]:
plot_TF(F_gf_beta, mean=True)

In [ ]:
plot_TF(F_sf_alpha, mean=True)

In [ ]:
plot_TF(F_sf_beta, mean=True)

In [ ]:
window = 1 
overlap = 0.9
settings = ['T1P', 'T1Pn', 'T3P', 'T3Pn']
channel = 'C3'
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'M', 'Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
M_psds = {'Male, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'F', 'Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
F_psds = {'Female, ' + old_key: old_value for old_key, old_value in psds.items()}
FM_alpha = M_psds | F_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'M', 'Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
M_psds = {'Male, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'F', 'Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
F_psds = {'Female, ' + old_key: old_value for old_key, old_value in psds.items()}
FM_Theta = M_psds | F_psds

In [ ]:
plot_TF(FM_T1Ps_alpha,mean=True)


In [ ]:
plot_TF(FM_T1Ps_Beta,mean=True)

In [ ]:
plot_TF(FM_T1Ps_Theta,mean=True)

In [ ]:
window = 1 # What should window size be?
overlap = 0.9
settings = ['T3P', 'T3Pn']
channel = 'C3'
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'M', 'Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
M_psds = {'Male, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'F', 'Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
F_psds = {'Female, ' + old_key: old_value for old_key, old_value in psds.items()}
FM_T3Ps_alpha = M_psds | F_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'M', 'Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
M_psds = {'Male, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'F', 'Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
F_psds = {'Female, ' + old_key: old_value for old_key, old_value in psds.items()}
FM_T3Ps_Theta = M_psds | F_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'M', 'Epochs'], channel=channel, settings=settings, band = [4,8], window=3/4, overlap = overlap)
M_psds = {'Male, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Gender'] == 'F', 'Epochs'], channel=channel, settings=settings, band = [4,8], window=3/4, overlap = overlap)
F_psds = {'Female, ' + old_key: old_value for old_key, old_value in psds.items()}
FM_T3Ps_Beta = M_psds | F_psds

In [ ]:
plot_TF(FM_T3Ps_alpha,mean=True)

In [ ]:
plot_TF(FM_T3Ps_Beta,mean=True)

In [ ]:
plot_TF(FM_T3Ps_Theta, mean = True)

In [ ]:
overlap = 0.9
settings = ['T1P', 'T1Pn', 'T3P', 'T3Pn']
channel = 'C3'
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, settings=settings, band = [8,12], window=3/8, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_alpha = F_psds | nF_psds
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'Yes', 'Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
F_psds = {'Friend, ' + old_key: old_value for old_key, old_value in psds.items()}
psds = get_psds_for_channel(participants_data=epochs_df.loc[epochs_df['Friend'] == 'No', 'Epochs'], channel=channel, settings=settings, band = [12,35], window=3/12, overlap = overlap)
nF_psds = {'Not friend, ' + old_key: old_value for old_key, old_value in psds.items()}
F_beta = F_psds | nF_psds

# Power graphs

In [ ]:
psds_list = list(Alpha_psds.values())[0] 
segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
power_T1P = segment_0_3.mean(axis=1)
psds_list = list(Alpha_psds.values())[1] 
segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
power_T1Pn = segment_0_3.mean(axis=1)
psds_list = list(Alpha_psds.values())[2] 
segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
power_T3P = segment_0_3.mean(axis=1)
psds_list = list(Alpha_psds.values())[3] 
segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
power_T3Pn = segment_0_3.mean(axis=1)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, ttest_ind

# Build DataFrame
df = pd.concat([
    pd.DataFrame({'Power': power_T1Pn, 'Condition': 'T1Pn'}),
    pd.DataFrame({'Power': power_T1P, 'Condition': 'T1P'}),
    pd.DataFrame({'Power': power_T3P, 'Condition': 'T3P'}),
    pd.DataFrame({'Power': power_T3Pn, 'Condition': 'T3Pn'})
], ignore_index=True)

# Run t-tests
p_val_t1p_vs_t1pn = ttest_rel(power_T1P, power_T1Pn).pvalue
p_val_t1p_vs_t3p  = ttest_rel(power_T1P, power_T3P).pvalue
p_val_t3p_vs_t3pn  = ttest_rel(power_T3P, power_T3Pn).pvalue

# Plot
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x='Condition', y='Power', inner='box', cut=0)
sns.swarmplot(data=df, x='Condition', y='Power', color='k', alpha=0.3, size=3)

# Annotate p-values
y_max = df['Power'].max()
offset = 0.05
line_height = y_max + offset
text_offset = offset - 0.01

# T1P vs T1Pn
p1_text = f"p = {p_val_t1p_vs_t1pn:.1e}"
plt.plot([0, 0, 1, 1], [line_height, line_height + 0.01, line_height + 0.01, line_height], lw=1.5, color='k')
plt.text(0.5, line_height + text_offset, p1_text, ha='center')

# T1P vs T3P
line_height_2 = line_height - 0.4
p2_text = f"p = {p_val_t1p_vs_t3p:.2f}"
plt.plot([1, 1, 2, 2], [line_height_2, line_height_2 + 0.01, line_height_2 + 0.01, line_height_2], lw=1.5, color='k')
plt.text(1.5, line_height_2 + text_offset, p2_text, ha='center')

# T3P vs T3Pn
line_height_3 = line_height - 0.2
p3_text = f"p = {p_val_t3p_vs_t3pn:.1e}"
plt.plot([2, 2, 3, 3], [line_height_3, line_height_3 + 0.01, line_height_3 + 0.01, line_height_3], lw=1.5, color='k')
plt.text(2.5, line_height_3 + text_offset, p3_text, ha='center')


# Final plot touches
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.title('Power distribution across participants (0-3s), Alpha band')
plt.ylabel('Relative Power')
plt.tight_layout()
plt.show()


In [ ]:
paired_data = [Alpha_psds, Beta_psds]
p_values_paired = {}
for psds in paired_data:
    powers = {}
    conditions = []
    for (condition, psd) in psds.items():
        psds_list = psd
        segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
        powers[condition] = segment_0_3.mean(axis=1)
        conditions.append(condition)
    p_values_paired[(conditions[0], conditions[1])] = ttest_rel(powers[conditions[0]], powers[conditions[1]]).pvalue
    p_values_paired[(conditions[0], conditions[2])] = ttest_rel(powers[conditions[0]], powers[conditions[2]]).pvalue
    p_values_paired[(conditions[2], conditions[3])] = ttest_rel(powers[conditions[2]], powers[conditions[3]]).pvalue
    p_values_paired[(conditions[1], conditions[3])] = ttest_rel(powers[conditions[1]], powers[conditions[3]]).pvalue
p_values_paired

In [ ]:
unpaired_data_FM = [FM_alpha, FM_Theta]
p_values_unpaired_FM = {}
for psds in unpaired_data_FM:
    powers = {}
    conditions = []
    for (condition, psd) in psds.items():
        psds_list = psd
        segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
        powers[condition] = segment_0_3.mean(axis=1)
        conditions.append(condition)
    p_values_unpaired_FM[(conditions[0], conditions[4])] = ttest_ind(powers[conditions[0]], powers[conditions[4]], equal_var=False).pvalue
    p_values_unpaired_FM[(conditions[1], conditions[5])] = ttest_ind(powers[conditions[1]], powers[conditions[5]], equal_var=False).pvalue
    p_values_unpaired_FM[(conditions[2], conditions[6])] = ttest_ind(powers[conditions[2]], powers[conditions[6]], equal_var=False).pvalue
    p_values_unpaired_FM[(conditions[3], conditions[7])] = ttest_ind(powers[conditions[3]], powers[conditions[7]], equal_var=False).pvalue
p_values_unpaired_FM

In [ ]:
unpaired_data_F = [F_alpha, F_beta]
p_values_unpaired_F = {}
for psds in unpaired_data_F:
    powers = {}
    conditions = []
    for (condition, psd) in psds.items():
        psds_list = psd
        segment_0_3 = psds_list[:, int(len(psds_list[0])/12):int(len(psds_list[0])/2 + len(psds_list[0])/12) + 1]
        powers[condition] = segment_0_3.mean(axis=1)
        conditions.append(condition)
    p_values_unpaired_F[(conditions[0], conditions[4])] = ttest_ind(powers[conditions[0]], powers[conditions[4]], equal_var=False).pvalue
    p_values_unpaired_F[(conditions[1], conditions[5])] = ttest_ind(powers[conditions[1]], powers[conditions[5]], equal_var=False).pvalue
    p_values_unpaired_F[(conditions[2], conditions[6])] = ttest_ind(powers[conditions[2]], powers[conditions[6]], equal_var=False).pvalue
    p_values_unpaired_F[(conditions[3], conditions[7])] = ttest_ind(powers[conditions[3]], powers[conditions[7]], equal_var=False).pvalue
p_values_unpaired_F

In [ ]:

ttest_rel(power_T1P, power_T1Pn).pvalue